In [1]:
import json
from pathlib import Path
from pprint import pprint

from IPython.display import HTML, Markdown, display

import altair as alt
import polars as pl

TIMELINE_WIDTH = 1_000


batches = Path("/Volumes/dsa-db-data/data")
with open(batches / "meta.json", mode="r", encoding="utf8") as file:
    meta = json.load(file)

frame = pl.DataFrame(
    [dict(date=k) | dict(v) for k, v in meta.items() if k != "summary"]
).with_columns(
    pl.col("date").str.to_date("%Y-%m-%d"),
    (pl.col("batch_rows") / pl.col("total_rows") * 100).alias("protection_of_minors_pct"),
    (pl.col("batch_rows_with_keywords") / pl.col("batch_rows") * 100).alias("batch_keywords_pct"),
    (pl.col("total_rows_with_keywords") / pl.col("total_rows") * 100).alias("total_keywords_pct"),
)

start_date, stop_date, batch_rows, batch_memory, batch_pct, total_rows = (
    frame.select(
        pl.col("date").min().alias("min"),
        pl.col("date").max().alias("max"),
        pl.col("batch_rows").sum(),
        pl.col("batch_memory").sum(),
        pl.col("protection_of_minors_pct").mean(),
        pl.col("total_rows").sum(),
    ).row(0)
)
batch_keywords_pct, total_keywords_pct = (
    frame.select(
        pl.col("batch_keywords_pct").mean(),
        pl.col("total_keywords_pct").mean(),
    ).row(0)
)

if 1_000_000_000 < batch_memory:
    batch_memory = batch_memory / 1_000_000_000
    unit = "GB"
else:
    batch_memory = batch_memory / 1_000_000
    unit = "MB"

display(HTML("<h1>EU DSA SoR Category: Protection of Minors<h1>"))

summary = {
    "Dates":
        f"__{start_date} to {stop_date}__ (inclusive)",
    "Rows with _Protection of Minors_ category":
        f"{batch_rows:,} out of {total_rows:,}, i.e., __{batch_pct:.1f}% of all rows__",
    "Rows with _Protection of Minors_ category and keywords":
        f"__{batch_keywords_pct:.1f}% of rows in category__ "
        f"(versus {total_keywords_pct:.1f}% across all categories)",
    "Total bytes for in-memory frames":
        f"__{batch_memory:,.1f} {unit}__",
}
widths = [max(len(k) for k in summary.keys()), max(len(v) for v in summary.values())]
summary_md = (
    f"|{'Description':<{widths[0]}}|{'Value':<{widths[1]}}|\n"
    + f"|{'-' * widths[0]}|{'-' * widths[1]}|\n"
    + "".join([f"|{k:<{widths[0]}}|{v:<{widths[1]}}|\n" for k, v in summary.items()])
)

schema_md = (
    f"|{'Column':<30}|{'Type':<20}|\n|{'-' * 30}|{'-' * 20}|\n"
    + "".join([f"|{str(key):<30}|{str(value):<20}|\n" for key, value in frame.schema.items()])
)




display(Markdown(f"""
### Summary
{summary_md}

### Schema
{schema_md}
"""))


### Summary
|Description                                           |Value                                                           |
|------------------------------------------------------|----------------------------------------------------------------|
|Dates                                                 |__2023-09-25 to 2024-05-07__ (inclusive)                        |
|Rows with _Protection of Minors_ category             |31,421,853 out of 10,785,175,497, i.e., __0.5% of all rows__    |
|Rows with _Protection of Minors_ category and keywords|__2.7% of rows in category__ (versus 5.6% across all categories)|
|Total bytes for in-memory frames                      |__28.9 GB__                                                     |


### Schema
|Column                        |Type                |
|------------------------------|--------------------|
|date                          |Date                |
|batch_count                   |Int64               |
|total_rows                    |Int64               |
|total_rows_with_keywords      |Int64               |
|batch_rows                    |Int64               |
|batch_rows_with_keywords      |Int64               |
|batch_memory                  |Int64               |
|protection_of_minors_pct      |Float64             |
|batch_keywords_pct            |Float64             |
|total_keywords_pct            |Float64             |



In [2]:
f1 = frame.select(
    pl.col("date"),
    pl.col("batch_rows") / 1_000,
    pl.col("batch_memory") / 1_000_000,
)

timeline1 = (alt
    .Chart(f1, title="SoRs with Protection of Minors")
    .mark_bar(tooltip=True)
    .encode(
        alt.X("date:T"),
        alt.Y("batch_rows:Q").title("thousand rows"),
    )
    .properties(width=TIMELINE_WIDTH)
    .interactive()
)

In [3]:
f2 = frame.select(
    pl.col("date"),
    (pl.col("batch_rows") / pl.col("total_rows") * 100).alias("protection_of_minors")
)

timeline2 = (
    alt.Chart(f2, title="Percent Fraction of SoRs with Protection of Minors").mark_bar(tooltip=True).encode(
        alt.X("date:T"),
        alt.Y("protection_of_minors:Q").title("percent"),
    )
    .properties(width=TIMELINE_WIDTH)
    .interactive()
)

In [4]:
f3 = frame.select(
    pl.col("date"),
    pl.col("batch_keywords_pct").alias("protection_of_minors"),
    pl.col("total_keywords_pct").alias("total"),
).unpivot(
    index=["date"],
    on=["protection_of_minors", "total"],
    variable_name="kind",
    value_name="pct",
)

timeline3 = (
    alt.Chart(f3, title="Percentage Fraction of SoRs with Keywords").mark_bar(tooltip=True).encode(
        alt.X("date:T"),
        alt.Y("pct:Q").title("percent"),
        alt.Color("kind:N"),
        order=alt.Order("kind", sort="ascending")
    )
    .properties(width=TIMELINE_WIDTH)
    .interactive()
)

In [5]:
timelines = [timeline1, timeline2, timeline3]
graph = alt.vconcat(*timelines)
graph

alt.VConcatChart(...)